# MerLog Chatbot — сравнение моделей

тут сравниваю TF-IDF плюс LogReg и DistilBERT на одном датасете чтобы понять стоит ли тащить трансформер в продакшн или классики хватит

датасет: Bitext Customer Support, отфильтровал до 6 логистических интентов, получилось 5985 примеров

## 1. Грузим данные и обучаем TF-IDF плюс LogReg

In [ ]:
import sys
sys.path.append('../src')

import time
import pandas as pd
from preprocessing import load_processed
from intent_classifier import train, save_model, predict

df = load_processed()
print(f"Dataset: {len(df)} rows, {df['intent'].nunique()} intents")
print(df.groupby('intent').size())

In [ ]:
start = time.time()
pipe, tfidf_metrics, _ = train(df)
tfidf_train_time = time.time() - start
print(f"TF-IDF + LogReg training: {tfidf_train_time:.2f}s")
print(f"Accuracy: {tfidf_metrics['accuracy']:.4f}")
print(f"F1 macro: {tfidf_metrics['f1_macro']:.4f}")

In [ ]:
start = time.time()
for _ in range(100):
    predict(pipe, "where is my shipment MRL-2024-8831")
tfidf_infer_time = (time.time() - start) / 100 * 1000
print(f"TF-IDF inference: {tfidf_infer_time:.2f} ms/query")

## 2. Обучаем DistilBERT

In [ ]:
from transformer_classifier import train_transformer, predict_transformer

start = time.time()
model, tokenizer, bert_metrics = train_transformer(df, epochs=3, batch_size=16)
bert_train_time = time.time() - start
print(f"DistilBERT training: {bert_train_time:.2f}s")
print(f"Accuracy: {bert_metrics['accuracy']:.4f}")
print(f"F1 macro: {bert_metrics['f1_macro']:.4f}")

In [ ]:
start = time.time()
for _ in range(50):
    predict_transformer("where is my shipment MRL-2024-8831", model, tokenizer)
bert_infer_time = (time.time() - start) / 50 * 1000
print(f"DistilBERT inference: {bert_infer_time:.2f} ms/query")

## 3. Сравнение

In [ ]:
comparison = pd.DataFrame({
    'Model': ['TF-IDF + LogReg', 'DistilBERT'],
    'Accuracy': [tfidf_metrics['accuracy'], bert_metrics['accuracy']],
    'F1 macro': [tfidf_metrics['f1_macro'], bert_metrics['f1_macro']],
    'Train time (s)': [tfidf_train_time, bert_train_time],
    'Inference (ms)': [tfidf_infer_time, bert_infer_time],
})
comparison

## 4. Вывод

DistilBERT даёт чуть выше точность, около 0.17%, но для продакшена выбрал TF-IDF плюс LogReg и вот почему:

- скорость: TF-IDF на порядки быстрее, для real-time чат бота это критично, клиент не будет ждать 15 мс когда можно за меньше 1 мс
- объяснимость: веса фичей можно посмотреть и показать стейкхолдерам MerLog почему бот принял такое решение
- простота: не нужен GPU, модель весит килобайты а не гигабайты, деплой проще
- разница в точности: 0.17% не стоит той сложности которую тащит трансформер

а confidence-роутер компенсирует чуть меньшую точность тем что неуверенные ответы уходит к оператору